Explainable Multi-Agent Reinforcement Learning for Autonomous Cyber Defense: A POSG Framework with SHAP-Driven Policy Interpretation
## Source Code & Environment Repository

**Environment:** Partially Observable Stochastic Game (POSG) (`CyberSecEnv`)
**Algorithms:** Double Deep Q-Network (DQN) & Proximal Policy Optimization (PPO)
**Training Paradigm:** Asymmetric Self-Play with Opponent Pooling

This notebook contains the complete source code required to reproduce the environment, training loops, baseline evaluations, and SHAP (Explainable AI) analysis presented in the manuscript.

# Install required dependencies
!pip install -q gymnasium==1.0.0 stable-baselines3 shimmy shap

In [ ]:
import os
import copy
import json
import random
import warnings
from collections import defaultdict

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import shap

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN, PPO

# Configuration for plot rendering
matplotlib.rcParams['figure.max_open_warning'] = 50
warnings.filterwarnings('ignore')

In [ ]:
# ==========================================
# 1. Configuration Constants & Hyperparameters
# ==========================================

# -- Network Topology --
N_PORTS        = 10
MAX_STEPS      = 200

# -- Traffic & IP Generation --
IP_NORMAL_POOL = 1000
IP_ATT_START   = 2000
N_ACTIVE_MIN   = 20
N_ACTIVE_MAX   = 40
LAMBDA_TOTAL   = 50
FP_NOISE_PROB  = 0.01

# -- Network Anomaly Mechanics --
MAX_RATE       = 25.0
ANOMALY_WINDOW = 50
ANOMALY_DECAY  = 0.95
MAX_ANOMALY    = 20.0

# -- Defensive Thresholds --
R_LIMIT        = 5
BLOCK_DURATION = 5

# -- Exploitation Mechanics --
BURST_SIZE     = 10
T_MIN          = 100
T_MAX          = 500
VULN_PROB      = 0.40
IP_COOLDOWN    = 10

# -- Deception Mechanics --
P_DETECT       = 0.60

# -- Reward Formulation --
R_EXPLOIT_WIN  = 100.0
R_TRAP_HIT     = 50.0
R_SCAN_COST    = 0.1
R_EXPLOIT_COST = 0.2
R_CHANGE_IP    = 10.0
R_FP_PENALTY   = 0.3
R_TRAP_COST    = 5.0
R_RATELIMIT_IP = 3.0
R_CLOSE_ATT    = 30.0
R_CLOSE_DEF    = 60.0
R_AVAIL_BONUS  = 0.025
R_DEF_WAIT     = 0.2

# -- Training Protocol --
# Note: Set N_CYCLES = 150 and N_SEEDS = 5 to fully replicate the manuscript.
# Reduced here to allow for feasible local demonstration execution.
N_CYCLES       = 50
STEPS_PER_CYCLE= 10_000
N_SEEDS        = 1
POOL_SIZE      = 5
POOL_SAVE_FREQ = 10

# -- Derived Action/Observation Dimensions --
ATT_ACTIONS    = 2 * N_PORTS + 3
DEF_ACTIONS    = 4 * N_PORTS + 2
ATT_OBS        = 3 * N_PORTS + 4
DEF_OBS        = 4 * N_PORTS + 3

def enforce_reproducibility(seed: int):
    """Secures random seeds across all libraries for reproducible experiments."""
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

In [ ]:
# ==========================================
# 2. Base Environment Formulation (POSG)
# ==========================================

class CyberSecEnv(gym.Env):
    """
    Partially Observable Stochastic Game (POSG) simulating network intrusion and defense.
    Enforces a strict two-phase execution pipeline to prevent turn-order desynchronization.
    """
    metadata = {"render_modes": ["human"]}

    def __init__(self, n=N_PORTS, max_steps=MAX_STEPS, p_detect=P_DETECT, t_min=T_MIN, t_max=T_MAX):
        super().__init__()
        self.n = n
        self.max_steps = max_steps
        self.p_detect = p_detect
        self.t_min = t_min
        self.t_max = t_max
        
        self.action_space_att = spaces.Discrete(2*n + 3)
        self.action_space_def = spaces.Discrete(4*n + 2)
        self.observation_space_att = spaces.Box(0., 1., (3*n + 4,), np.float32)
        self.observation_space_def = spaces.Box(0., 1., (4*n + 3,), np.float32)
        
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            np.random.seed(seed)
            random.seed(seed)
            
        self.current_step = 0
        
        # Initialize Background Traffic State
        na = np.random.randint(N_ACTIVE_MIN, N_ACTIVE_MAX + 1)
        self.active_ips = np.random.choice(range(1, IP_NORMAL_POOL + 1), size=na, replace=False)
        
        # Initialize Attacker State
        self.att_ip = IP_ATT_START
        self.steps_since_change = IP_COOLDOWN
        self.exploit_cnt = np.zeros(self.n)
        self.scanned = np.zeros(self.n, dtype=bool)
        self.last_att_act = 0
        
        # Initialize Environmental Vulnerabilities
        self.vuln = np.array([random.random() < VULN_PROB for _ in range(self.n)])
        self.T_p = np.array([random.randint(self.t_min, self.t_max) if v else float('inf') for v in self.vuln], dtype=np.float32)
        
        # Initialize Defender State
        self.traps = np.zeros(self.n, dtype=bool)
        self.limited = np.zeros(self.n, dtype=bool)
        self.closed = np.zeros(self.n, dtype=bool)
        self.ip_scores = {}
        self.ip_seen = {}
        self.blocked = {}
        self.last_def_act = 0
        self.fp_penalty_acc = 0.0
        
        # Traffic Counters
        self.port_req = np.zeros(self.n)
        self.port_sus = np.zeros(self.n)
        self.port_legit = np.zeros(self.n)
        
        # Episode Terminal Flags
        self._exploit_ok = False
        self._trap_hit = False
        
        self.last_obs_att = self._generate_attacker_obs()
        self.last_obs_def = self._generate_defender_obs()
        
        return {"attacker": self.last_obs_att, "defender": self.last_obs_def}, {}

    def _simulate_background_traffic(self):
        """Generates Poisson-distributed legitimate traffic and calculates false-positive noise."""
        self.fp_penalty_acc = 0.0
        total_requests = int(np.random.poisson(LAMBDA_TOTAL))
        if total_requests == 0: 
            return
            
        ips = np.random.choice(self.active_ips, size=total_requests)
        ports = np.random.randint(0, self.n, size=total_requests)
        
        for ip, j in zip(ips, ports):
            ip, j = int(ip), int(j)
            
            if self.closed[j]:
                continue
                
            if ip in self.blocked:
                self.fp_penalty_acc += R_FP_PENALTY
                continue
                
            if self.limited[j] and self.port_req[j] >= R_LIMIT:
                self.fp_penalty_acc += R_FP_PENALTY
                continue
                
            self.port_req[j] += 1
            self.port_legit[j] += 1
            self.ip_seen[ip] = self.current_step
            
            if np.random.random() < FP_NOISE_PROB:
                self.port_sus[j] += 1
                self.ip_scores[ip] = min(self.ip_scores.get(ip, 0) + 0.1, MAX_ANOMALY)

    def attacker_phase(self, action: int) -> float:
        """Phase 1: Applies attacker action and simulates network traffic."""
        self.port_req.fill(0)
        self.port_sus.fill(0)
        self.port_legit.fill(0)
        self._exploit_ok = False
        self._trap_hit = False
        
        self.last_att_act = action
        self.steps_since_change += 1
        reward = 0.0
        n = self.n
        
        if 0 <= action < n:  # Scan Action
            j = action
            if self.att_ip not in self.blocked and not self.closed[j]:
                self.scanned[j] = True
                self.port_req[j] += 1
                self.port_sus[j] += 1
                self.ip_scores[self.att_ip] = min(self.ip_scores.get(self.att_ip, 0) + 1.0, MAX_ANOMALY)
                self.ip_seen[self.att_ip] = self.current_step
            reward -= R_SCAN_COST
            
        elif n <= action < 2*n:  # Exploit Action
            j = action - n
            if self.att_ip not in self.blocked and not self.closed[j]:
                burst = min(BURST_SIZE, R_LIMIT) if self.limited[j] else BURST_SIZE
                self.exploit_cnt[j] += burst
                self.port_req[j] += burst
                self.port_sus[j] += burst
                self.ip_scores[self.att_ip] = min(self.ip_scores.get(self.att_ip, 0) + burst * 0.5, MAX_ANOMALY)
                self.ip_seen[self.att_ip] = self.current_step
                
                if self.traps[j]:
                    self._trap_hit = True
                elif self.exploit_cnt[j] >= self.T_p[j]:
                    self._exploit_ok = True
            reward -= R_EXPLOIT_COST
            
        elif action == 2*n:  # Change IP Action
            if self.steps_since_change >= IP_COOLDOWN:
                self.att_ip += 1
                self.exploit_cnt.fill(0)
                self.steps_since_change = 0
                reward -= R_CHANGE_IP
                
        elif action == 2*n + 2:  # Cancel Action
            reward -= R_EXPLOIT_COST
            
        self._simulate_background_traffic()
        self.last_obs_def = self._generate_defender_obs()
        return reward

    def defender_phase(self, action: int):
        """Phase 2: Applies defensive action, calculates terminal states, and finalizes rewards."""
        self.last_def_act = action
        current_fp_penalty = self.fp_penalty_acc
        r_def = -current_fp_penalty
        r_att_extra = 0.0
        self.fp_penalty_acc = 0.0
        n = self.n
        
        if action == 0:  # Wait Action
            r_def -= R_DEF_WAIT
            
        elif action == 1:  # Rate-Limit Top Anomalous IP
            active = {ip: s for ip, s in self.ip_scores.items() if self.current_step - self.ip_seen.get(ip, 0) <= ANOMALY_WINDOW}
            if active:
                top_ip = max(active, key=active.get)
                self.blocked[top_ip] = BLOCK_DURATION
            r_def -= R_RATELIMIT_IP
            
        elif 2 <= action < n + 2:  # Rate-Limit Port
            j = action - 2
            if not self.closed[j]:
                self.limited[j] = True
                r_def -= R_RATELIMIT_IP
                
        elif n + 2 <= action < 2*n + 2:  # Deploy Trap
            j = action - (n + 2)
            if not self.closed[j]:
                self.traps[j] = True
                r_def -= R_TRAP_COST
                
        elif 2*n + 2 <= action < 3*n + 2:  # Close Port
            j = action - (2*n + 2)
            if not self.closed[j]:
                self.closed[j] = True
                r_def -= R_CLOSE_DEF
                r_att_extra -= R_CLOSE_ATT
                
        elif 3*n + 2 <= action < 4*n + 2:  # Restore Port
            j = action - (3*n + 2)
            self.closed[j] = False
            self.limited[j] = False

        # Apply Service Availability Bonus
        r_def += np.sum(np.where(~self.closed & ~self.limited, R_AVAIL_BONUS,
                                 np.where(~self.closed & self.limited, R_AVAIL_BONUS * 0.5, 0.0)))
                                 
        # Terminal State Evaluation
        terminated = False
        win_condition = "timeout"
        
        if self._exploit_ok:
            r_att_extra += R_EXPLOIT_WIN
            r_def -= R_EXPLOIT_WIN
            terminated = True
            win_condition = "attacker"
        elif self._trap_hit:
            r_att_extra -= R_TRAP_HIT
            r_def += R_TRAP_HIT
            terminated = True
            win_condition = "defender"
        elif all(self.closed):
            terminated = True
            win_condition = "all_closed"

        self._apply_temporal_decay()
        self.current_step += 1
        truncated = self.current_step >= self.max_steps
        
        self.last_obs_att = self._generate_attacker_obs()
        self.last_obs_def = self._generate_defender_obs()
        
        info = {
            "win": win_condition, 
            "step": self.current_step,
            "ports_closed": sum(self.closed),
            "ports_limited": sum(self.limited),
            "traps_active": sum(self.traps),
            "fp_events": int(current_fp_penalty / R_FP_PENALTY)
        }
        
        return r_def, r_att_extra, terminated, truncated, info

    def _apply_temporal_decay(self):
        """Applies exponential decay to IP anomaly scores."""
        for ip in list(self.ip_scores):
            self.ip_scores[ip] *= ANOMALY_DECAY
            if self.ip_scores[ip] < 0.01:
                del self.ip_scores[ip]
                self.ip_seen.pop(ip, None)
        for ip in list(self.blocked):
            self.blocked[ip] -= 1
            if self.blocked[ip] <= 0:
                del self.blocked[ip]

    def _generate_attacker_obs(self):
        o = np.zeros(self.observation_space_att.shape, dtype=np.float32)
        n = self.n
        o[:n] = np.where(self.scanned, np.where(self.vuln, 1.0, 0.5), 0.0)
        
        valid_Tp = ~np.isinf(self.T_p)
        o[n:2*n][valid_Tp] = np.clip(self.exploit_cnt[valid_Tp] / self.T_p[valid_Tp], 0.0, 1.0)
        
        trap_mask = self.traps & (np.random.random(n) < self.p_detect)
        o[2*n:3*n] = np.where(trap_mask, 1.0, 0.0)
        
        o[3*n]   = 0.0 if self.att_ip in self.blocked else 1.0
        o[3*n+1] = min(self.steps_since_change / IP_COOLDOWN, 1.0)
        o[3*n+2] = self.current_step / self.max_steps
        o[3*n+3] = self.last_att_act / float(ATT_ACTIONS)
        return o

    def _generate_defender_obs(self):
        o = np.zeros(self.observation_space_def.shape, dtype=np.float32)
        n = self.n
        o[:n] = np.clip(self.port_req / MAX_RATE, 0.0, 1.0)
        
        req_mask = self.port_req > 0
        o[n:2*n][req_mask] = np.clip(self.port_sus[req_mask] / self.port_req[req_mask], 0.0, 1.0)
        
        o[2*n:3*n] = np.where(self.traps, 1.0, 0.0)
        o[3*n:4*n] = np.where(self.closed, 0.0, np.where(self.limited, 0.5, 1.0))
        
        o[4*n]   = min(max(self.ip_scores.values(), default=0) / MAX_ANOMALY, 1.0)
        o[4*n+1] = self.current_step / self.max_steps
        o[4*n+2] = self.last_def_act / float(DEF_ACTIONS)
        return o

In [ ]:
# ==========================================
# 3. Stable-Baselines3 Wrappers
# ==========================================

class AttackerEnv(gym.Env):
    """Exposes the POSG strictly from the Attacker's perspective for SB3."""
    def __init__(self, base_env):
        self.env = base_env
        self.action_space = base_env.action_space_att
        self.observation_space = base_env.observation_space_att
        self.opponent = None
        
    def reset(self, seed=None, options=None):
        obs, _ = self.env.reset(seed=seed)
        return obs["attacker"], {}
        
    def step(self, action):
        r_att = self.env.attacker_phase(int(action))
        d_act = 0
        if self.opponent is not None:
            d_act, _ = self.opponent.predict(self.env.last_obs_def, deterministic=False)
            
        r_def, r_att2, term, trunc, info = self.env.defender_phase(int(d_act))
        return self.env.last_obs_att, r_att + r_att2, term, trunc, info

class DefenderEnv(gym.Env):
    """Exposes the POSG strictly from the Defender's perspective for SB3."""
    def __init__(self, base_env):
        self.env = base_env
        self.action_space = base_env.action_space_def
        self.observation_space = base_env.observation_space_def
        self.opponent = None
        
    def reset(self, seed=None, options=None):
        obs, _ = self.env.reset(seed=seed)
        if self.opponent is not None:
            a_act, _ = self.opponent.predict(self.env.last_obs_att, deterministic=False)
        else:
            a_act = np.random.randint(0, ATT_ACTIONS)
            
        self.env.attacker_phase(int(a_act))
        return self.env.last_obs_def, {}
        
    def step(self, action):
        r_def, r_att2, term, trunc, info = self.env.defender_phase(int(action))
        done = term or trunc
        if done:
            return self.env.last_obs_def, r_def, term, trunc, info
            
        a_act = 2 * self.env.n + 1
        if self.opponent is not None:
            a_act, _ = self.opponent.predict(self.env.last_obs_att, deterministic=False)
            
        self.env.attacker_phase(int(a_act))
        return self.env.last_obs_def, r_def, term, trunc, info

In [ ]:
# ==========================================
# 4. Heuristic & Rule-Based Baselines
# ==========================================

class RandomAgent:
    def __init__(self, n): 
        self.n = n
    def predict(self, obs, deterministic=False):
        return np.random.randint(0, self.n), None

class GreedyAttacker:
    def __init__(self, N): 
        self.N = N
    def predict(self, obs, deterministic=True):
        prog = obs[self.N : 2*self.N]
        if obs[3*self.N] < 0.5 and obs[3*self.N + 1] >= 1.0:
            return 2*self.N, None
            
        anom = obs[2*self.N : 3*self.N]
        best = int(np.argmax(prog))
        if anom[best] > 0.7: 
            return 2*self.N + 2, None
        return self.N + best, None

class ScriptedAttacker:
    def __init__(self, N):
        self.N = N
        self._step = 0
        self._scanned = set()
        
    def reset(self):
        self._step = 0
        self._scanned = set()
        
    def predict(self, obs, deterministic=True):
        self._step += 1
        if self._step % 15 == 0: 
            return 2*self.N, None
            
        for i in range(self.N):
            if i not in self._scanned:
                self._scanned.add(i)
                return i, None
                
        best = int(np.argmax(obs[:self.N]))
        return (self.N + best, None) if obs[best] > 0.5 else (2*self.N + 1, None)

class ThresholdDefender:
    """Simulates a static SOC threshold algorithm (e.g., Fail2Ban)."""
    def __init__(self, N, b=0.6, a=0.5, c=0.85):
        self.N = N
        self.b = b
        self.a = a
        self.c = c
        
    def predict(self, obs, deterministic=True):
        for j in range(self.N):
            if obs[self.N+j] > self.c and obs[3*self.N+j] > 0:
                return 2*self.N + 2 + j, None
                
        if obs[4*self.N] > self.b: 
            return 1, None
            
        for j in range(self.N):
            if obs[self.N+j] > self.a and obs[3*self.N+j] == 1.0:
                return 2 + j, None
                
        return 0, None

class ReactiveDefender:
    """Simulates an active incident response escalation ladder."""
    def __init__(self, N, th=0.4):
        self.N = N
        self.th = th
        
    def predict(self, obs, deterministic=True):
        for j in range(self.N):
            if obs[self.N+j] < self.th: 
                continue
            if obs[2*self.N+j] < 0.5: 
                return self.N + 2 + j, None
            if obs[3*self.N+j] == 1.0: 
                return 2 + j, None
            if obs[3*self.N+j] == 0.5: 
                return 2*self.N + 2 + j, None
                
        return 0, None

In [ ]:
# ==========================================
# 5. Opponent Pooling Mechanism (70/20/10)
# ==========================================

import tempfile

class OpponentPool:
    """Maintains a historical distribution of policies to prevent catastrophic forgetting."""
    def __init__(self, role, max_size=POOL_SIZE):
        self.role = role
        self.paths = []
        self.max_size = max_size

    def save(self, model):
        path = tempfile.mktemp(suffix='.zip')
        model.save(path)
        self.paths.append(path)
        if len(self.paths) > self.max_size:
            old = self.paths.pop(0)
            if os.path.exists(old): 
                os.remove(old)

    def sample(self, current_model, model_cls, env, n):
        r = random.random()
        
        # 10% Probability: Static Heuristic Baseline
        if r < 0.10:                                    
            if self.role == 'defender':
                return random.choice([RandomAgent(DEF_ACTIONS), ThresholdDefender(n)])
            else:
                return random.choice([RandomAgent(ATT_ACTIONS), GreedyAttacker(n)])
                
        # 20% Probability: Historical Snapshot
        elif r < 0.30 and len(self.paths) > 0:          
            try:
                loaded = model_cls.load(random.choice(self.paths), env=env, device='auto')
                loaded.policy.set_training_mode(False)
                return loaded
            except Exception:
                return current_model
                
        # 70% Probability: Current Policy
        return current_model

In [ ]:
# ==========================================
# 6. Evaluation & Metrics Tracking
# ==========================================

class MetricsLogger:
    def __init__(self):
        self.data = defaultdict(list)

    def log(self, cycle, env, att_model, def_model, n_eval=20):
        att_r, def_r, ep_len = [], [], []
        att_w, def_w, to = 0, 0, 0
        avail, port_closes, traps_triggered, fp_total = [], [], 0, 0
        att_acts = np.zeros(ATT_ACTIONS)
        def_acts = np.zeros(DEF_ACTIONS)

        for _ in range(n_eval):
            if hasattr(att_model, 'reset'): att_model.reset()
            if hasattr(def_model, 'reset'): def_model.reset()
            obs, _ = env.reset()
            done = False
            total_a, total_d, steps = 0, 0, 0
            ep_avail, ep_closes, ep_fps = 0.0, 0, 0
            
            while not done:
                a, _ = att_model.predict(env.last_obs_att, deterministic=True)
                r_att1 = env.attacker_phase(int(a))

                d, _ = def_model.predict(env.last_obs_def, deterministic=True)
                r_def, r_att2, term, trunc, info = env.defender_phase(int(d))

                att_acts[int(a)] += 1
                def_acts[int(d)] += 1
                if 2*env.n+2 <= int(d) < 3*env.n+2: 
                    ep_closes += 1

                ep_fps += info.get("fp_events", 0)
                total_a += (r_att1 + r_att2)
                total_d += r_def
                steps += 1
                done = term or trunc

                avail_this_step = sum(
                    1.0 if not env.closed[j] and not env.limited[j]
                    else 0.5 if env.limited[j] and not env.closed[j] else 0.0
                    for j in range(env.n)) / env.n
                ep_avail += avail_this_step
                
                if done:
                    w = info.get("win", "timeout")
                    if w == "attacker": att_w += 1
                    elif w == "defender":
                        def_w += 1
                        traps_triggered += 1
                    else: to += 1
                    
            att_r.append(total_a)
            def_r.append(total_d)
            ep_len.append(steps)
            avail.append(ep_avail / max(steps, 1))
            port_closes.append(ep_closes / max(steps, 1))
            fp_total += ep_fps

        att_acts /= max(att_acts.sum(), 1)
        def_acts /= max(def_acts.sum(), 1)

        self.data['cycle'].append(cycle)
        self.data['att_reward_mean'].append(np.mean(att_r))
        self.data['att_reward_std'].append(np.std(att_r))
        self.data['def_reward_mean'].append(np.mean(def_r))
        self.data['def_reward_std'].append(np.std(def_r))
        self.data['att_win_rate'].append(att_w / n_eval)
        self.data['def_win_rate'].append(def_w / n_eval)
        self.data['timeout_rate'].append(to / n_eval)
        self.data['mean_ep_len'].append(np.mean(ep_len))
        self.data['availability'].append(np.mean(avail))
        self.data['trap_trigger_rate'].append(traps_triggered / n_eval)
        self.data['port_close_freq'].append(np.mean(port_closes))
        self.data['false_positive_rate'].append(fp_total / max(np.sum(ep_len), 1))
        
        self.data['att_entropy'].append(-np.sum(att_acts[att_acts>0] * np.log(att_acts[att_acts>0] + 1e-10)))
        self.data['def_entropy'].append(-np.sum(def_acts[def_acts>0] * np.log(def_acts[def_acts>0] + 1e-10)))
        self.data['att_action_dist'].append(att_acts.copy())
        self.data['def_action_dist'].append(def_acts.copy())

        print(f"  Cycle {cycle:03d}: AttR={np.mean(att_r):.1f} | DefR={np.mean(def_r):.1f} | "
              f"WR(A/D/T)={att_w/n_eval:.2f}/{def_w/n_eval:.2f}/{to/n_eval:.2f} | "
              f"EpLen={np.mean(ep_len):.0f}")

def evaluate_pairing(env, att, dfn, n_episodes=100):
    """Executes a rigorous evaluation phase for a specific agent pairing."""
    att_w, def_w, to = 0, 0, 0
    att_r, def_r, ep_lens = [], [], []
    fp_events, avail_sum = 0, 0.0
    
    for _ in range(n_episodes):
        if hasattr(att, 'reset'): att.reset()
        if hasattr(dfn, 'reset'): dfn.reset()
        obs, _ = env.reset()
        done = False
        ra, rd, steps = 0, 0, 0
        ep_avail = 0.0
        
        while not done:
            a, _ = att.predict(env.last_obs_att, deterministic=True)
            r_att1 = env.attacker_phase(int(a))
            
            d, _ = dfn.predict(env.last_obs_def, deterministic=True)
            r_def, r_att2, t, trunc, info = env.defender_phase(int(d))
            
            ra += (r_att1 + r_att2)
            rd += r_def
            steps += 1
            fp_events += info.get("fp_events", 0)
            
            ep_avail += sum(1.0 if not env.closed[j] and not env.limited[j]
                            else 0.5 if env.limited[j] and not env.closed[j] else 0.0
                            for j in range(env.n)) / env.n
            done = t or trunc
            
            if done:
                w = info.get("win", "timeout")
                if w == "attacker": att_w += 1
                elif w == "defender": def_w += 1
                else: to += 1
                
        att_r.append(ra)
        def_r.append(rd)
        ep_lens.append(steps)
        avail_sum += ep_avail / max(steps, 1)
        
    return {
        "att_wr": att_w/n_episodes, 
        "def_wr": def_w/n_episodes,
        "timeout": to/n_episodes,
        "att_r_mean": np.mean(att_r), 
        "att_r_std": np.std(att_r),
        "def_r_mean": np.mean(def_r), 
        "def_r_std": np.std(def_r),
        "ep_len": np.mean(ep_lens),
        "fp_rate": fp_events/max(sum(ep_lens), 1),
        "availability": avail_sum/n_episodes
    }

def run_full_evaluation(trained_agents, n=N_PORTS):
    """Benchmarks trained models against established heuristic baselines."""
    env = CyberSecEnv(n=n)
    agents = {
        'random_att':    RandomAgent(ATT_ACTIONS),
        'greedy_att':    GreedyAttacker(n),
        'scripted_att':  ScriptedAttacker(n),
        'random_def':    RandomAgent(DEF_ACTIONS),
        'threshold_def': ThresholdDefender(n),
        'reactive_def':  ReactiveDefender(n),
    }
    agents.update(trained_agents)

    pairings = [
        ('random_att','random_def'),
        ('dqn_att','threshold_def'), 
        ('ppo_att','threshold_def'),
        ('greedy_att','dqn_def'), 
        ('greedy_att','ppo_def'),
        ('dqn_att','ppo_def'), 
        ('ppo_att','dqn_def'),
    ]
    
    results = {}
    print(f"\n{'='*65}\nBaseline Evaluation Matrix (100 Episodes)\n{'='*65}")
    for an, dn in pairings:
        if an not in agents or dn not in agents:
            continue
        res = evaluate_pairing(env, agents[an], agents[dn])
        results[f"{an}_vs_{dn}"] = res
        print(f"{an:>14s} vs {dn:<14s} | "
              f"A-WR:{res['att_wr']*100:5.1f}%  D-WR:{res['def_wr']*100:5.1f}%  "
              f"T-OR:{res['timeout']*100:5.1f}% | EpLen:{res['ep_len']:4.0f}")
    return results

In [ ]:
# ==========================================
# 7. Asymmetric Self-Play Training Loop
# ==========================================

def train_self_play(att_algo='DQN', def_algo='PPO', n_cycles=N_CYCLES, seed=42, n=N_PORTS, save_path=None):
    enforce_reproducibility(seed)
    base = CyberSecEnv(n=n)
    att_env = AttackerEnv(base)
    def_env = DefenderEnv(base)

    def instantiate_model(algo, env, is_attacker):
        if algo == 'DQN':
            return DQN("MlpPolicy", env, learning_rate=1e-4,
                       buffer_size=100000, learning_starts=1000,
                       batch_size=512, target_update_interval=500,
                       exploration_fraction=0.5,
                       exploration_initial_eps=1.0,
                       exploration_final_eps=0.05,
                       gamma=0.99,
                       policy_kwargs=dict(net_arch=[256,256]),
                       device="auto", verbose=0)
        else:
            return PPO("MlpPolicy", env, learning_rate=3e-4,
                       n_steps=2048, batch_size=64, n_epochs=10,
                       gamma=0.99, gae_lambda=0.95,
                       clip_range=0.2, ent_coef=0.01, vf_coef=0.5,
                       max_grad_norm=0.5,
                       policy_kwargs=dict(net_arch=dict(pi=[256,256], vf=[256,256])),
                       device="auto", verbose=0)

    att_model = instantiate_model(att_algo, att_env, True)
    def_model = instantiate_model(def_algo, def_env, False)
    
    att_pool = OpponentPool('attacker')
    def_pool = OpponentPool('defender')
    logger = MetricsLogger()

    print(f"\n{'='*60}")
    print(f"Training: {att_algo} Attacker vs {def_algo} Defender | Seed: {seed}")
    print(f"{'='*60}")

    for cycle in range(n_cycles):
        if cycle > 0 and cycle % POOL_SAVE_FREQ == 0:
            att_pool.save(att_model)
            def_pool.save(def_model)

        # Attacker Training Phase
        opp = def_pool.sample(def_model, type(def_model), def_env, n)
        att_env.opponent = opp
        att_model.learn(total_timesteps=STEPS_PER_CYCLE, reset_num_timesteps=(cycle == 0))

        # Defender Training Phase
        opp = att_pool.sample(att_model, type(att_model), att_env, n)
        def_env.opponent = opp
        def_model.learn(total_timesteps=STEPS_PER_CYCLE, reset_num_timesteps=(cycle == 0))

        # Validation Evaluation
        if cycle % 5 == 0 or cycle == n_cycles - 1:
            eval_env = CyberSecEnv(n=n)
            logger.log(cycle, eval_env, att_model, def_model)

        # Checkpoint Saving
        if save_path and cycle % 50 == 0 and cycle > 0:
            att_model.save(f'{save_path}/{att_algo}_att_c{cycle}_s{seed}')
            def_model.save(f'{save_path}/{def_algo}_def_c{cycle}_s{seed}')
            with open(f'{save_path}/metrics_c{cycle}_s{seed}.json', 'w') as f:
                json.dump({k: [x.tolist() if isinstance(x, np.ndarray) else x for x in v] 
                           for k, v in logger.data.items()}, f)

    return att_model, def_model, logger

In [ ]:
# ==========================================
# 8. Visualization & XAI Functions
# ==========================================

def plot_training_curves(loggers, labels, title_prefix=""):
    """Generates the performance grid mapped to the manuscript figures."""
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    plot_cfg = [
        ('att_reward_mean', 'def_reward_mean', 'Mean Reward'),
        ('att_win_rate',    'def_win_rate',     'Win Rate'),
        ('mean_ep_len',     None,               'Episode Length'),
        ('att_entropy',     'def_entropy',       'Action Entropy'),
        ('timeout_rate',    None,               'Timeout Rate'),
        ('availability',    None,               'System Availability'),
        ('false_positive_rate', None,           'False Positive Rate'),
        ('trap_trigger_rate', None,             'Trap Trigger Rate'),
        ('port_close_freq', None,               'Port Closure Freq'),
    ]
    
    for idx, (key1, key2, title) in enumerate(plot_cfg):
        ax = axes[idx // 3, idx % 3]
        for logger, label in zip(loggers, labels):
            d = logger if isinstance(logger, dict) else logger.data
            c = d['cycle']
            if key1 in d: 
                ax.plot(c, d[key1], label=f'{label} Att' if key2 else label)
            if key2 and key2 in d: 
                ax.plot(c, d[key2], label=f'{label} Def', linestyle='--')
        ax.set_title(f'{title_prefix}{title}')
        ax.set_xlabel('Cycle')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
        
    plt.tight_layout()
    if save_name:
        import os
        os.makedirs(os.path.dirname(save_name), exist_ok=True)
        plt.savefig(save_name, bbox_inches='tight')
    plt.show()

def plot_multiseed(all_loggers, title_prefix="", save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for key in ['att_reward_mean', 'def_reward_mean']:
        data = np.array([l[key] for l in all_loggers])
        mean, std = data.mean(0), data.std(0)
        cycles = all_loggers[0]['cycle']
        label = 'Attacker' if 'att' in key else 'Defender'
        color = 'red' if 'att' in key else 'blue'
        axes[0].plot(cycles, mean, label=label, color=color)
        axes[0].fill_between(cycles, mean-std, mean+std, alpha=0.2, color=color)
    axes[0].set_title(f'{title_prefix}Reward (mean±std, {len(all_loggers)} seeds)')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    
    for key in ['att_win_rate', 'def_win_rate']:
        data = np.array([l[key] for l in all_loggers])
        mean, std = data.mean(0), data.std(0)
        cycles = all_loggers[0]['cycle']
        label = 'Attacker' if 'att' in key else 'Defender'
        color = 'red' if 'att' in key else 'blue'
        axes[1].plot(cycles, mean, label=label, color=color)
        axes[1].fill_between(cycles, mean-std, mean+std, alpha=0.2, color=color)
    axes[1].set_title('Win Rate'); axes[1].legend(); axes[1].grid(alpha=0.3)
    
    data = np.array([l['mean_ep_len'] for l in all_loggers])
    mean, std = data.mean(0), data.std(0)
    axes[2].plot(cycles, mean, color='green')
    axes[2].fill_between(cycles, mean-std, mean+std, alpha=0.2, color='green')
    axes[2].set_title('Episode Length'); axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        import os
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight')
        print(f'Saved plot: {save_path}')
    plt.show()


# -- SHAP Analysis Setup --
ATT_FEATURES = (
    [f"vuln_p{i}" for i in range(N_PORTS)] +
    [f"prog_p{i}" for i in range(N_PORTS)] +
    [f"anom_p{i}" for i in range(N_PORTS)] +
    ["ip_health", "steps_since_chg", "ep_progress", "prev_action"]
)

DEF_FEATURES = (
    [f"traffic_p{i}" for i in range(N_PORTS)] +
    [f"anom_ratio_p{i}" for i in range(N_PORTS)] +
    [f"trap_p{i}" for i in range(N_PORTS)] +
    [f"avail_p{i}" for i in range(N_PORTS)] +
    ["max_ip_score", "ep_progress", "prev_action"]
)

def collect_obs(model, env_wrapper, n_samples=1000):
    obs_list = []
    obs, _ = env_wrapper.reset()
    for _ in range(n_samples):
        obs_list.append(obs)
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, _ = env_wrapper.step(action)
        if done: 
            obs, _ = env_wrapper.reset()
    return np.array(obs_list)

def shap_analysis(model, env_wrapper, feature_names, title="", n_samples=500, save_name=None):
    """Deconstructs the learned neural policies using SHapley Additive exPlanations."""
    obs_data = collect_obs(model, env_wrapper, n_samples)
    obs_tensor = torch.FloatTensor(obs_data).to(model.device)
    
    try:
        net = torch.nn.Sequential(
            model.policy.mlp_extractor.policy_net,
            model.policy.action_net
        )
    except AttributeError:
        try:
            net = model.q_net.q_net  # Structure for SB3 DQN
        except:
            print("Error: Could not extract PyTorch network for SHAP execution.")
            return
            
    bg = obs_tensor[:100]
    explainer = shap.DeepExplainer(net, bg)
    sv = explainer.shap_values(obs_tensor[:min(300, len(obs_tensor))], check_additivity=False)

    if isinstance(sv, list):
        sv = np.abs(np.array(sv)).mean(axis=0)

    plt.figure(figsize=(12, 8))
    shap.summary_plot(sv, obs_tensor[:min(300, len(obs_tensor))].cpu().numpy(),
                      feature_names=feature_names, plot_type="bar", show=False)
    plt.title(f'SHAP Action-Feature Interactions \u2014 {title}')
    plt.tight_layout()


In [ ]:
# ==========================================
# 9. Main Execution Block
# ==========================================

if __name__ == "__main__":
    # Ensure local directory exists for checkpoint saving
    SAVE_DIR = './cyberrl_results/'
    import os
    import json
    os.makedirs(SAVE_DIR, exist_ok=True)

    CONFIGS = [
        ('DQN', 'PPO'),   # Cross-play agents (Primary Setup)
        ('DQN', 'DQN'),   # Baseline comparison 1
        ('PPO', 'PPO')    # Baseline comparison 2
    ]
    
    all_results = {}
    trained_agents = {}
    
    # 1. Execute Training Phase for All Configurations
    for att_algo, def_algo in CONFIGS:
        tag = f"{att_algo}_att_vs_{def_algo}_def"
        all_results[tag] = []
        print(f"\n--- Initiating Suite: {tag} ---")
        
        for seed in range(N_SEEDS):
            sp = f'{SAVE_DIR}/{tag}/seed_{seed}'
            os.makedirs(sp, exist_ok=True)
            att, defn, logger = train_self_play(
                att_algo=att_algo, 
                def_algo=def_algo, 
                n_cycles=N_CYCLES, 
                seed=seed, 
                save_path=sp
            )
            all_results[tag].append(logger.data)
            att.save(f'{sp}/final_att')
            defn.save(f'{sp}/final_def')
            
            # Keep latest seed agents for evaluation matrix & SHAP (for demonstration)
            if att_algo == 'DQN' and def_algo == 'PPO':
                trained_agents['dqn_att'] = att
                trained_agents['ppo_def'] = defn
            elif att_algo == 'DQN' and def_algo == 'DQN':
                trained_agents['dqn_def'] = defn
            elif att_algo == 'PPO' and def_algo == 'PPO':
                trained_agents['ppo_att'] = att

    print("\nAll training iterations complete!")

    # 2. Render Training Dynamics Plot
    print("\nRendering Learning Dynamics for all configurations...")
    for tag, loggers in all_results.items():
        if loggers:
            plot_multiseed(loggers, title_prefix=f"{tag.replace('_', ' ')} ", 
                           save_path=f"{SAVE_DIR}/{tag}_training_curves.pdf")

    # 3. Execute Formal Evaluation Matrix
    print("\nRunning Full Evaluation Matrix...")
    eval_results = run_full_evaluation(trained_agents)
    with open(f"{SAVE_DIR}/evaluation_matrix.json", "w") as f:
        json.dump(eval_results, f, indent=2)

    # 4. Execute SHAP (XAI) Analysis (using DQN att vs PPO def models)
    print("\nInitializing SHAP DeepExplainer Analysis...")
    base_env = CyberSecEnv(n=N_PORTS)
    
    att_env_eval = AttackerEnv(base_env)
    att_env_eval.opponent = trained_agents['ppo_def']
    
    def_env_eval = DefenderEnv(base_env)
    def_env_eval.opponent = trained_agents['dqn_att']

    print("Generating Explanation for DQN Attacker...")
    shap_analysis(trained_agents['dqn_att'], att_env_eval, ATT_FEATURES, "DQN Attacker", n_samples=300, save_name=f"{SAVE_DIR}/shap_DQN_Attacker.pdf")

    print("Generating Explanation for PPO Defender...")
    shap_analysis(trained_agents['ppo_def'], def_env_eval, DEF_FEATURES, "PPO Defender", n_samples=300, save_name=f"{SAVE_DIR}/shap_PPO_Defender.pdf")
    
    print("\nExperiment Suite Execution Complete.")
